# Stagnation detection in practice

## Solving a linear system using the conjugate gradient method

### Motivation

The conjugate gradient method is a way of solving a linear system $Ax = b$, where $A$ is a symmetric positive definite $n$ by $n$ matrix, in at most $n$ steps. This is done by computing steps in search directions $p_i$, where $p_i^\mathrm{T} A p_j = 0$ for $i \neq j$. The algorithm is as follows: first, define an initial guess $x_0$ to the linear system $Ax = b$ given $A$ and $b$, define the residual $r_0 = Ax_0 - b$, and define the first search direction as $p_0 = r_0$. Then each iteration performs
\begin{align*}
	{x}_{i+1} &= {x}_i + \frac{{r}_i^\mathrm{T}{r}_i}{{p}^\mathrm{T}_i{Ap}_i}{p}_i\\
	{r}_{i+1} &= {r}_i - \frac{{r}_i^\mathrm{T}{r}_i}{{p}^\mathrm{T}_i{Ap}_i}{Ap}_i\\
	{p}_{i+1} &= {r}_{i+1}+\frac{{r}_{i+1}^\mathrm{T}{r}_{i+1}}{{r}_{i}^\mathrm{T}{r}_{i}}{p}_i.
\end{align*}
until termination.

### Example

An implementation of the conjugate gradient method in Julia:

In [1]:
import LinearAlgebra: dot, norm

"""
Solve Ax = b using the conjugate gradient method.
"""
function cgsolve(A::Matrix{T}, b::Vector{T}, x_init::Vector{T}) where {T <: AbstractFloat}

    # initialisations
    maxIterations::Int = 1000
    tol::T = 1e-6

    n::Int = size(A)[1]

    xPrev::Vector{T} = x_init
    rPrev::Vector{T} = b - A*xPrev
    pPrev::Vector{T} = rPrev  
    r::Vector{T} = zeros(n)
    p::Vector{T} = zeros(n)
    x::Vector{T} = zeros(n)

    α::T = 0
    β::T = 0

    for iter in 1:maxIterations
        # calculate new values
        α = dot(rPrev, rPrev)/dot(pPrev, A*pPrev)
        x = xPrev + α*pPrev
        r = rPrev - α*A*pPrev

        if norm(r, 2) < tol
            break
        end

        # calculate new conjugate vector
        β = dot(r, r) / dot(rPrev, rPrev)
        p = r + β*pPrev

        # step placeholder variables
        pPrev = p
        rPrev = r
        xPrev = x
    end

    return x
end

cgsolve

For fun, let's make a symmetric positive definite matrix generator

In [2]:
import LinearAlgebra: tril
import Random: Xoshiro

function rand_symposdef(rng, T::DataType, n::Int)
    nums = Int.(rand(rng, UInt64, n, n) .% 4) .+ 1
    dens = Int.(rand(rng, UInt64, n, n) .% 8) .+ 1

    for i in 1:n
        for j in 1:n
            d = dens[i, j]
            if d >= 6
                dens[i, j] = 8
            elseif d >= 3
                dens[i, j] = 4
            end
        end
    end

    L = tril(nums .// dens)

    return T.(L * L')
end

rand_symposdef(T::DataType, n::Int) = rand_symposdef(Xoshiro(), T, n)

rand_symposdef (generic function with 2 methods)

Generate a symmetric positive definite system:

In [3]:
rng = Xoshiro(1)
A = rand_symposdef(rng, Float32, 8)

8×8 Matrix{Float32}:
 0.0625   0.09375   0.125    0.25    0.0625   0.1875   0.125    0.0625
 0.09375  0.390625  0.4375   1.875   0.34375  0.40625  0.3125   0.21875
 0.125    0.4375    1.5      3.0     0.625    1.5      0.875    0.5
 0.25     1.875     3.0     11.5625  2.1875   2.6875   2.5      1.625
 0.0625   0.34375   0.625    2.1875  0.6875   0.875    0.75     0.625
 0.1875   0.40625   1.5      2.6875  0.875    2.5      1.875    1.4375
 0.125    0.3125    0.875    2.5     0.75     1.875    2.20312  1.51562
 0.0625   0.21875   0.5      1.625   0.625    1.4375   1.51562  1.40625

In [4]:
rng = Xoshiro(2)
b = rand(rng, Float32, 8)

8-element Vector{Float32}:
 0.088336766
 0.0022505522
 0.20259368
 0.68074745
 0.29953098
 0.73533
 0.6909602
 0.80673164

Solve in single precision and observe the impact of backward error for both our method and Julia's solve using `\`:

In [5]:
x_init = ones(Float32, 8)

xtrue = A\b

x = cgsolve(A, b, x_init)

[b A*x A*xtrue]

8×3 Matrix{Float32}:
 0.0883368   0.0883339   0.0883466
 0.00225055  0.00222778  0.00225449
 0.202594    0.202545    0.202606
 0.680747    0.680603    0.680725
 0.299531    0.299515    0.29953
 0.73533     0.735275    0.735352
 0.69096     0.690918    0.690979
 0.806732    0.806702    0.806732

To use stagnation detection, all we need to do is change the operating type to a subtype of `FloatSD`. Operations with vectors and matrices will behave as expected.

In [6]:
using StagnationDetection

A_stag = Float32SD.(A)
b_stag = Float32SD.(b)
x_init_stag = Float32SD.(x_init)

x_stag = cgsolve(A_stag, b_stag, x_init_stag)

8-element Vector{Float32SD}:
  330.87845f0
 -300.533f0
  -87.25064f0
   69.56155f0
  -30.174871f0
   53.79144f0
  -59.95494f0
    6.3001347f0

Verify that this is close to the real solution:

In [7]:
getfield.(x_stag, :value) ≈ xtrue

true

We can then identify stagnation for different entries in the solution obtained by the conjugate gradient method.

In [8]:
adds = getfield.(x_stag, :additions)

uvals = getfield.(x_stag, :u)

stag = getfield.(x_stag, :stagnation)

println("Addition preservation of bits:")
println(uvals ./ adds, "\n")

println("Stagnation observed?")
println(any(stag .!= 0))

println("Length of total stagnation for each:")
println(Int.(stag))

report(x_stag[1])

Addition preservation of bits:
[10.576923076923077, 11.0, 11.5, 11.538461538461538, 11.692307692307692, 12.0, 12.192307692307692, 13.692307692307692]

Stagnation observed?
true
Length of total stagnation for each:
[4, 3, 2, 4, 1, 1, 2, 1]

--------------- SUMMARY ----------------
Floating point additions performed: 26.
Total stagnation of length 4.
The condition for total is u = 0 and for the smaller addend to be rounded off completely in a given addition.
See paper: Analysis and Detection of Stagnation in Floating-Point Summation,
Mantas Mikaitis and William Woolfenden, 2026 (in progress)

-------- PROCESSING OF RESULTS ---------
u is the total number of bits which were not shifted off in the addition.
Total number of bits processed u = 275.
Average uᵢ = 10.576923076923077.

------------ END OF SUMMARY ------------


We can get stagnation data for individual entries, but we can also generate data for the stagnation profile of the whole problem.

Feel free to change the seeds for the RNG and see how stagnation is affected. We have found that for similar size problems, stagnation usually occurs to some extent.

One problem here is that we are generating entirely positive matrices and vectors of specific distributions. Until we try the same problem with a selection of test matrices we cannot conclude the likelihood of stagnation in the conjugate gradient method.